In [0]:
%run /Users/nethumgimsara605@gmail.com/ecommerce-lakehouse-databricks-repo/01-ingestion/setup-storage-connection

In [0]:
from pyspark.sql.functions import count, row_number, col
from pyspark.sql.window import Window

In [0]:
bronze_orders = spark.read.format("parquet").load("abfss://bronze@ecommercelakehouse01.dfs.core.windows.net/orders/")
silver_customers = spark.read.format("delta").load("abfss://silver@ecommercelakehouse01.dfs.core.windows.net/customers/").filter("is_current = true")
silver_products = spark.read.format("delta").load("abfss://silver@ecommercelakehouse01.dfs.core.windows.net/products/")

print(f"Bronze orders: {bronze_orders.count()}")
print(f"Current silver customers: {silver_customers.count()}")
print(f"Silver products: {silver_products.count()}")

In [0]:
duplicaate_check = bronze_orders.groupby("order_id").agg(count("*").alias("cnt")).filter("cnt > 1")
print(f"Duplicate order_ids found: {duplicaate_check.count()}")

window_spec = Window.partitionBy("order_id").orderBy("order_timestamp")
deduped_orders = bronze_orders.withColumn("row_num", row_number().over(window_spec)) \
    .filter("row_num = 1") \
    .drop("row_nm")

print(f"Rows before dedup: {bronze_orders.count()}")
print(f"Rows after dedup: {deduped_orders.count()}")

In [0]:
orders_with_invalid_customer = deduped_orders.join(
    silver_customers.select("customer_id"),
    on="customer_id",
    how="left_anti"
)
orders_with_invalid_customer.write.format("delta").mode("append").save("abfss://silver@ecommercelakehouse01.dfs.core.windows.net/_quarantine/orders_invalid_customer_fk")

print(f"Orders with inavlid customer_id: {orders_with_invalid_customer.count()}")

In [0]:
orders_with_invalid_product = deduped_orders.join(
    silver_products.select("product_id"),
    on="product_id",
    how="left_anti"
)

orders_with_invalid_product.write.format("delta").mode("append").save("abfss://silver@ecommercelakehouse01.dfs.core.windows.net/_quarantine/orders_invalid_product_fk")

print(f"Orders with inavlid product_id: {orders_with_invalid_product.count()}")

In [0]:
orders_with_bad_values = deduped_orders.filter(
    "quantity <= 0 OR unit_price <= 0 OR total_amount <= 0"
)

orders_with_bad_values.write.format("delta").mode("append").save("abfss://silver@ecommercelakehouse01.dfs.core.windows.net/_quarantine/orders_bad_values")

print(f"Orders with ivalid quantity/price/amount: {orders_with_bad_values.count()}")


In [0]:
bad_order_ids = orders_with_invalid_customer.select("order_id") \
    .union(orders_with_invalid_product.select("order_id")) \
    .union(orders_with_bad_values.select("order_id")) \
    .distinct()
cleaned_orders = deduped_orders.join(bad_order_ids, on="order_id", how="left_anti")

print(f"Total bad/quarantined order_ids: {bad_order_ids.count()}")
print(f"Deduped orders: {deduped_orders.count()}")
print(f"Final cleaned orders: {cleaned_orders.count()}")

In [0]:
cleaned_orders.write.format("delta").mode("overwrite").save("abfss://silver@ecommercelakehouse01.dfs.core.windows.net/orders/")

In [0]:
silver_orders_check = spark.read.format("delta").load("abfss://silver@ecommercelakehouse01.dfs.core.windows.net/orders/")
silver_orders_check.printSchema()
print(f"Total rows in silver orders: {silver_orders_check.count()}")